# 🛰️ RSICC (Remote Sensing Image Change Captioning) - Google Colab Eğitim & Değerlendirme Notebook

Bu notebook, **RSICCformer** modelini Google Colab GPU (T4 / V100 / A100) üzerinde uçtan uca çalıştırmak için hazırlanmıştır.

### 📌 İşlem Adımları:
1. **GPU & Java Kontrolü** (METEOR metriği için Java gereklidir)
2. **Google Drive Bağlama** (Kodlar, Veri Seti ve Çıktılar için)
3. **Kod Dosyalarının Açılması ve Çalışma Dizinine Geçiş**
4. **Gereksinimlerin Kurulması** (`requirements.txt`)
5. **Veri Setinin İndirilmesi ve Zipten Çıkarılması** (`Levir_CC_dataset`)
6. **Veri Ön İşleme** (`create_input_files.py`)
7. **Model Eğitimi** (`train.py` - Checkpoint'ler doğrudan Google Drive'a kaydedilir)
8. **Test Değerlendirmesi** (`eval.py` - BLEU, METEOR, ROUGE-L, CIDEr metrikleri)

## 1. Donanım (GPU) ve Java Kontrolü
> **Önemli:** Üst menüden **Düzenle (Edit) -> Not Defteri Ayarları (Notebook Settings)** kısmından **T4 GPU** seçildiğinden emin olun.

In [ ]:
# GPU bilgilerini kontrol edelim
!nvidia-smi

# METEOR değerlendirme metriği Java 11+ gerektirir (Colab'de varsayılan yüklüdür)
!java -version

## 2. Google Drive'ı Bağlama
Eğitilen model ağırlıklarının (`.pth.tar`) ve sonuçların bağlantı kopsa bile kaybolmaması için Google Drive'ınızı bağlıyoruz.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Kod Dosyalarını Açma & Çalışma Dizinine Geçiş
- Drive'ınıza yüklediğiniz kod zip dosyasının adını aşağıdaki `CODE_ZIP_PATH` değişkenine yazın.
- Kodlar Colab'in yerel hızlı SSD'sine (`/content/RSICC`) açılır.
- `%cd /content/RSICC` komutuyla çalışma dizinine geçildiği için **tüm Python importları yerelde olduğu gibi sorunsuz çalışır**.

In [ ]:
import os

# Drive'daki kod zip dosyanızın yolu (dosya adınız farklıysa güncelleyin):
CODE_ZIP_PATH = '/content/drive/MyDrive/RSICC.zip'

# Hedef proje klasörü
PROJECT_DIR = '/content/RSICC'
os.makedirs(PROJECT_DIR, exist_ok=True)

# Zip dosyasını çıkartalım
!unzip -q "{CODE_ZIP_PATH}" -d "{PROJECT_DIR}"

# Proje çalışma dizinine geçiyoruz
%cd {PROJECT_DIR}

# Dosyaları listeleyip kontrol edelim
!ls -la

## 4. Bağımlılıkların Kurulması
Güncel `requirements.txt` dosyasındaki kütüphaneleri yüklüyoruz.

In [ ]:
# Gerekli yardımcı kütüphaneleri kuralım
!pip install -r requirements.txt --quiet
!pip install -U --no-cache-dir gdown --quiet
print("Tüm bağımlılıklar başarıyla yüklendi!")

## 5. Veri Setini İndirme & Zipten Çıkarma
Aşağıdaki `DATASET_DRIVE_URL` alanına veri setinin bulunduğu Google Drive linkini yapıştırın.
`gdown` aracıyla dosya indirilip `Levir_CC_dataset/` klasörüne açılacaktır.

In [ ]:
# Veri setinin paylaşılan Google Drive linki (Burayı kendi linkinizle değiştirin):
DATASET_DRIVE_URL = "BURAYA_DATASET_DRIVE_LINKINI_YAPISTIRIN"

# Hedef dataset klasörü
DATASET_DIR = "/content/RSICC/Levir_CC_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

# Veri setini indirelim
!gdown "{DATASET_DRIVE_URL}" -O /content/dataset_download.zip

# İndirilen zipi Levir_CC_dataset içine açalım
!unzip -q /content/dataset_download.zip -d "{DATASET_DIR}"

# Dosya yapısını doğrulayalım (LevirCCcaptions.json ve images klasörü görünmelidir)
!ls -lh "{DATASET_DIR}"

## 6. Veri Setini Ön İşleme (`create_input_files.py`)
Resimler okunup 256x256 boyutuna getirilerek hızlı eğitim için `./data/` klasörüne HDF5 ve JSON olarak yazılır.

In [ ]:
# Veri ön işlemeyi başlat
!python create_input_files.py

# Oluşan dosyaları kontrol edelim
!ls -lh ./data/

## 7. Modeli Eğitme (`train.py`)
- Checkpoint kayıt yolu (`--savepath`), **Google Drive** üzerinde oluşturulan `RSICC_Outputs/models_checkpoint/` klasörüne yönlendirilmiştir.
- Colab T4 GPU için `--batch_size 16` ve `--workers 2` önerilir (eğitimi çok hızlandırır).

In [ ]:
# Checkpoint'lerin Drive'da kaydedileceği klasör
DRIVE_CHECKPOINT_DIR = "/content/drive/MyDrive/RSICC_Outputs/models_checkpoint/"
os.makedirs(DRIVE_CHECKPOINT_DIR, exist_ok=True)

# Eğitimi başlatıyoruz
!python train.py \
  --data_folder ./data/ \
  --data_name LEVIR_CC_5_cap_per_img_5_min_word_freq \
  --savepath "{DRIVE_CHECKPOINT_DIR}" \
  --epochs 40 \
  --batch_size 16 \
  --workers 2 \
  --encoder_lr 1e-4 \
  --decoder_lr 1e-4

## 8. Test Setinde Değerlendirme (`eval.py`)
Eğitilen en iyi model (`BEST_checkpoint_...pth.tar`) Google Drive'dan okunur, test seti üzerinde değerlendirilir ve sonuçlar Drive'a kopyalanır.

In [ ]:
# Değerlendirmeyi çalıştır (Drive'daki en iyi modeli kullanarak)
!python eval.py \
  --data_folder ./data/ \
  --data_name LEVIR_CC_5_cap_per_img_5_min_word_freq \
  --path "{DRIVE_CHECKPOINT_DIR}" \
  --Split TEST \
  --beam_size 1

# Değerlendirme JSON sonuçlarını Drive'a yedekleyelim
DRIVE_RESULTS_DIR = "/content/drive/MyDrive/RSICC_Outputs/eval_results/"
os.makedirs(DRIVE_RESULTS_DIR, exist_ok=True)
!cp -r ./eval_results_fortest/* "{DRIVE_RESULTS_DIR}"
print("Değerlendirme sonuçları başarıyla Google Drive'a yedeklendi!")

## 9. Örnek Resim Üzerinde Çıkarım Demo (`caption.py`)
Örnek resim çifti (`Example/A` ve `Example/B`) üzerinden eğitilen modelin ürettiği altyazıyı test edebilirsiniz.

In [ ]:
!python caption.py \
  --img_A ./Example/A/train_000016.png \
  --img_B ./Example/B/train_000016.png \
  --path "{DRIVE_CHECKPOINT_DIR}" \
  --data_folder ./data/ \
  --data_name LEVIR_CC_5_cap_per_img_5_min_word_freq

# Üretilen altyazıyı ekrana yazdır
!cat ./eval_results/*.json